# 00: Multi-Unit Raster Mosaicking (GDAL VRT)

This notebook demonstrates how to use the `landseg` raster mosaicking utility (`build_vrt_mosaic`) to assemble disparate raster tiles into a unified, reprojected **GDAL Virtual Raster (`.vrt`)** mosaic.

### Context & Motivation
In regional and provincial remote sensing workflows (e.g. Ontario Crown Forest Management Units), imagery and thematic landcover layers often span multiple UTM zones (e.g. UTM Zone 16N, 17N, 18N) or administrative unit boundaries. 

Rather than reserializing gigabytes of heavy imagery to disk, `build_vrt_mosaic`:
1. **Zero Pixel Duplication:** Assembles tiles purely in virtual XML metadata.
2. **Cross-Projection Reprojection:** Transparently warps tiles from differing source projections (e.g., UTM 16N/17N/18N) into the canonical Ontario provincial grid (`EPSG:3161` Ontario MNR Lambert).
3. **Categorical & Continuous Modes:** Preserves discrete integer class boundaries (via nearest-neighbor) or resamples continuous imagery (bilinear, cubic, lanczos).
4. **Direct Pipeline Integration:** The resulting `.vrt` file can be passed directly as an input layer into `dataset_manifest.json` for subsequent harmonization, tiling, and model training.

## 1. Environment Setup & Imports

If running on **Databricks**, uncomment and run the `%pip install -e ..` cell below.

In [ ]:
# Environment and path resolution
import os
import sys
import pathlib

repo_root = pathlib.Path('..').resolve()
sys.path.insert(0, str(repo_root / 'src'))

# Third-party imports
import numpy as np
import rasterio
import rasterio.crs
import rasterio.transform
import matplotlib.pyplot as plt

# Landseg Public API
import landseg.geopipe.harmonize as harmonize

print('Successfully imported landseg and dependencies!')

## 2. Generate Synthetic Cross-UTM Raster Tiles for Demonstration

To demonstrate bridging disparate projections, we generate two synthetic GeoTIFF tiles:
- **Tile A (West):** Projected in **UTM Zone 17N** (`EPSG:32617`).
- **Tile B (East):** Projected in **UTM Zone 18N** (`EPSG:32618`).

Both tiles represent adjacent imagery across a UTM zone boundary.

In [ ]:
demo_dir = repo_root / 'experiment' / 'demo_mosaic'
demo_dir.mkdir(parents=True, exist_ok=True)

# Create Tile A in UTM Zone 17N
tile_a_path = demo_dir / 'tile_a_utm17.tif'
crs_17 = rasterio.crs.CRS.from_string('EPSG:32617')
transform_17 = rasterio.transform.from_origin(500000.0, 5200000.0, 20.0, 20.0)
data_17 = (np.arange(100 * 100).reshape(1, 100, 100) % 250).astype(np.uint8)

with rasterio.open(
    tile_a_path, 'w', driver='GTiff',
    width=100, height=100, count=1, dtype='uint8',
    crs=crs_17, transform=transform_17, nodata=255
) as dst:
    dst.write(data_17)
    dst.set_band_description(1, 'spectral_band')

# Create Tile B in UTM Zone 18N
tile_b_path = demo_dir / 'tile_b_utm18.tif'
crs_18 = rasterio.crs.CRS.from_string('EPSG:32618')
transform_18 = rasterio.transform.from_origin(300000.0, 5200000.0, 20.0, 20.0)
data_18 = ((np.arange(100 * 100).reshape(1, 100, 100) + 50) % 250).astype(np.uint8)

with rasterio.open(
    tile_b_path, 'w', driver='GTiff',
    width=100, height=100, count=1, dtype='uint8',
    crs=crs_18, transform=transform_18, nodata=255
) as dst:
    dst.write(data_18)
    dst.set_band_description(1, 'spectral_band')

print(f'Generated Tile A: {tile_a_path} (CRS: {crs_17})')
print(f'Generated Tile B: {tile_b_path} (CRS: {crs_18})')

## 3. Build Continuous Imagery VRT Mosaic

We now invoke `harmonize.build_vrt_mosaic()` to merge both tiles into a single virtual raster reprojected to Ontario MNR Lambert (`EPSG:3161`).

Key options used:
- `target_crs='EPSG:3161'`
- `resampling_method='bilinear'`
- `target_resolution=20.0` (20-meter pixel resolution)

In [ ]:
output_vrt = demo_dir / 'mosaic_imagery.vrt'

result_path = harmonize.build_vrt_mosaic(
    input_paths=[tile_a_path, tile_b_path],
    output_path=output_vrt,
    target_crs='EPSG:3161',
    target_resolution=20.0,
    resampling_method='bilinear',
    is_categorical=False,
    relative_paths=True,
)

print(f'Master Mosaic Created at: {result_path}')

# Inspect VRT metadata
with rasterio.open(result_path) as ds:
    print(f'CRS: {ds.crs}')
    print(f'Dimensions (W x H): {ds.width} x {ds.height}')
    print(f'Band Count: {ds.count}')
    print(f'Band Descriptions: {ds.descriptions}')
    print(f'Bounds: {ds.bounds}')
    print(f'Pixel Resolution: ({ds.res[0]:.2f}m, {ds.res[1]:.2f}m)')

## 4. Build Categorical Landcover Classification Mosaic

For discrete label layers (e.g. Ontario Land Cover, FRI forest species classes):
- Set `is_categorical=True`.
- This strictly enforces **nearest-neighbor resampling** during reprojection, guaranteeing integer class IDs are never interpolated or blurred into invalid fractional values.

In [ ]:
# Generate discrete label tiles with class IDs [1, 2, 3, 4]
label_a = demo_dir / 'labels_a_utm17.tif'
label_b = demo_dir / 'labels_b_utm18.tif'

with rasterio.open(
    label_a, 'w', driver='GTiff',
    width=80, height=80, count=1, dtype='uint8',
    crs=crs_17, transform=transform_17, nodata=0
) as dst:
    dst.write(np.random.randint(1, 5, size=(1, 80, 80)).astype(np.uint8))

with rasterio.open(
    label_b, 'w', driver='GTiff',
    width=80, height=80, count=1, dtype='uint8',
    crs=crs_18, transform=transform_18, nodata=0
) as dst:
    dst.write(np.random.randint(1, 5, size=(1, 80, 80)).astype(np.uint8))

label_mosaic_vrt = demo_dir / 'mosaic_labels.vrt'
harmonize.build_vrt_mosaic(
    input_paths=[label_a, label_b],
    output_path=label_mosaic_vrt,
    target_crs='EPSG:3161',
    is_categorical=True,
    nodata=0,
    relative_paths=True,
)

with rasterio.open(label_mosaic_vrt) as ds:
    sample_data = ds.read(1, window=rasterio.windows.Window(0, 0, min(ds.width, 50), min(ds.height, 50)))
    unique_classes = np.unique(sample_data)
    print(f'Discrete classes present in mosaic: {unique_classes}')
    assert sample_data.dtype == np.uint8
    print('Class integrity verified: nearest-neighbor preserved integer types!')

## 5. Visualizing the Mosaic

We can read directly from the generated `.vrt` file with `rasterio` just like any standard GeoTIFF, and plot the composited spatial coverage.

In [ ]:
with rasterio.open(output_vrt) as src:
    # Read a decimated thumbnail for fast preview
    overview_factor = max(1, src.width // 200)
    preview = src.read(
        1,
        out_shape=(src.height // overview_factor, src.width // overview_factor),
        resampling=rasterio.enums.Resampling.bilinear
    )

plt.figure(figsize=(9, 5))
plt.title('Preview of Reprojected Mosaic (EPSG:3161)', fontsize=12, fontweight='bold')
im = plt.imshow(preview, cmap='viridis', origin='upper')
plt.colorbar(im, label='Digital Number (DN)')
plt.xlabel('Pixel Column (X)')
plt.ylabel('Pixel Row (Y)')
plt.tight_layout()
plt.show()

## 6. CLI Equivalent Usage

You can also run this exact workflow from the command line using the dedicated script under `scripts/`:

```bash
python scripts/build_raster_mosaic.py \
    experiment/demo_mosaic/*.tif \
    --output experiment/demo_mosaic/mosaic_cli.vrt \
    --target-crs EPSG:3161 \
    --resampling bilinear \
    --relative-paths
```

### Next Steps
Now that your regional tiles are assembled into unified VRT layers, proceed to **`01_data_preparation.ipynb`** to define your canonical World Grid, compile the dataset manifest, and run data harmonization and ingestion.